### Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['GROQ_API_KEY'] = os.getenv('LLM_API')

### Summarization Middleware

Automatically summarize the conversation history to fit within the token limit of the model. This is especially useful for long conversations where the history might exceed the model's context window.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

### MessageBased Summarization
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:llama-3.3-70b-versatile',
            trigger=("messages",10),
            keep=('messages',4)
        )
    ]
)

In [5]:
### Run with thread id
config = {
    "configurable":{
        "thread_id":'test-1'
    }
}

In [8]:
# Test questions
questions = [
    'What is 2+2 ?',
    'What is 10 X 5 ?',
    'What is 100 / 4 ?',
    'What is 3 X 3 ?',
    'What is 4 X 4 ?',
    'What is 15 - 7 ?',
]
for q in questions:
    response = agent.invoke({'messages':[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to ask the AI to perform simple arithmetic operations and receive the results.\n\n## SUMMARY\nThe user has asked the AI to perform several arithmetic operations, including addition, multiplication, and division. The AI has provided the correct results for each operation: 2+2=4, 10*5=50, and 100/4=25. The user has also asked for the result of 3*3, but the AI has not yet responded.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe AI should respond to the user's question about 3*3, providing the result of the multiplication operation.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='29a1f355-e02f-4c9b-8e72-da5bd8bd1faa'), AIMessage(content='3 * 3 = 9.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 120, 'total_tokens': 129, 'completion_time': 0.009118072, 'completion_tokens_

#### Token Size

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str)->str:
    '''Search hotels --> returns long response to use more tokens'''
    return f"""Hotels in city {city}:
1. Grand Hotel - 5 star, $350 / Night, spa, pool, gym
2. City Inn - 4 star, $100 / Night
3. Budget Stay - 3 star, $75 / Night
"""

agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:llama-3.3-70b-versatile",
        trigger = ('tokens',550),
        keep = ('tokens',200)
    )]
)
config= {'configurable':{'thread_id':'test-1'}}

# token count
def count_tokens(messages):
    total_chars = sum((len(str(m.content)) for m in messages))
    return total_chars//4

In [17]:
cities = ['Paris','London','New York','Singapore']
for city in cities:
    response = agent.invoke(
        {'messages':[HumanMessage(content=f'Find hotels in city {city}')]},
        config=config
    )
    tokens = count_tokens(response['messages'])
    print(f"{city} ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris ~138 tokens, 6 messages
[HumanMessage(content='Find hotels in city Paris', additional_kwargs={}, response_metadata={}, id='49dfff75-2ea9-46fe-9d48-f2e433ed841a'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dzhct4113', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 231, 'total_tokens': 246, 'completion_time': 0.048270386, 'completion_tokens_details': None, 'prompt_time': 0.014534853, 'prompt_tokens_details': None, 'queue_time': 0.165462919, 'total_time': 0.062805239}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f5a-a5f1-76e0-9690-439c4d2f5e43-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'dzhct4113', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

### Fraction

In [19]:
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str)->str:
    '''Search hotels --> returns long response to use more tokens'''
    return f"""Hotels in city {city}:
1. Grand Hotel - 5 star, $350 / Night, spa, pool, gym
2. City Inn - 4 star, $100 / Night
3. Budget Stay - 3 star, $75 / Night
"""

agent = create_agent(
    model='groq:llama-3.3-70b-versatile',
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model='groq:llama-3.3-70b-versatile',
        trigger=('fraction',0.005), # 0.005 of model token context window
        keep=('fraction',0.002)
    )]
)

config = {'configurable':{'thread_id':'test-1'}}

# total tokens
def count_tokens(messages):
    total = sum(len(str(m.content)) for m in messages)
    return total//4

cities = ['Paris','London','New York','Singapore','Japan','Pakistan']
for city in cities:
    response = agent.invoke(
        {'messages':[HumanMessage(content=f'Find hotels in city {city}')]},
        config=config
    )
    tokens = count_tokens(response['messages'])
    print(f"{city} ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris ~348 tokens, 10 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels in the city of Paris.\n\n## SUMMARY\nThe user requested a list of hotels in Paris, and the system provided a list of three hotels: Grand Hotel (5-star, $350/night), City Inn (4-star, $100/night), and Budget Stay (3-star, $75/night). The same list was provided multiple times due to repeated tool calls.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next step would be for the user to select a hotel from the provided list or to refine their search criteria to get more specific results. Alternatively, the user may want to compare the features and prices of the listed hotels to make an informed decision.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='a82d99dc-0553-49fa-abb5-8fd62bd40a58'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5wgktbygy', 'function': {'arguments': '{"city":"P

### Human in the Loop

In [59]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

def read_email_tool(email_id:str)->str:
    """Mock function to read email"""
    return f"Email content for id: {email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject {subject}"

In [60]:
agent  = create_agent(
    model='groq:llama-3.3-70b-versatile',
    tools=[send_email_tool,read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)
config = {'configurable':{"thread_id":"test-1"}}

result = agent.invoke(
    {'messages':[HumanMessage(content="Send email to john@test.com with subject as 'Hello' and body is 'How are you'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject as 'Hello' and body is 'How are you'", additional_kwargs={}, response_metadata={}, id='096b9d01-ecfa-4e84-bd49-9343ba219e2a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7k7xk0vys', 'function': {'arguments': '{"body":"How are you","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 303, 'total_tokens': 335, 'completion_time': 0.047805089, 'completion_tokens_details': None, 'prompt_time': 0.037964362, 'prompt_tokens_details': None, 'queue_time': 0.317874093, 'total_time': 0.085769451}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f86-71e8-7130-9e8b-a915a9c381ce-0', tool_calls=[{'name': 'send_email_tool', 'args':

In [61]:
# step 2 take decision
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused Approving!!!!")

    result = agent.invoke(
        Command(
            resume={
                'decisions':[
                    {'type':"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused Approving!!!!
Result: Email sent to john@test.com with subject Hello


### Reject

In [69]:
agent = create_agent(
    model='groq:llama-3.3-70b-versatile',
    checkpointer=InMemorySaver(),
    tools=[read_email_tool,send_email_tool],
    system_prompt="""
If a tool call is rejected by the user,
inform the user politely that the action was cancelled.
""",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)
config = {'configurable':{"thread_id":"test-reject"}}
result = agent.invoke(
    {'messages':[HumanMessage(content="Send email to john@test.com with subject as 'Hello' and body is 'How are you'")]},
    config=config
)

In [70]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused Approving!!!!")

    result = agent.invoke(
        Command(
            resume={
                'decisions':[
                    {'type':"reject"}
                ]
            }
        ),
        config=config
    )
    print(result)
    print(f"Result: {result['messages'][-1].content}")

Paused Approving!!!!
{'messages': [HumanMessage(content="Send email to john@test.com with subject as 'Hello' and body is 'How are you'", additional_kwargs={}, response_metadata={}, id='2cabecfe-de73-4647-b405-2706c73072e4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'eay0k296y', 'function': {'arguments': '{"body":"How are you","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 323, 'total_tokens': 355, 'completion_time': 0.055435842, 'completion_tokens_details': None, 'prompt_time': 0.043124083, 'prompt_tokens_details': None, 'queue_time': 0.291406395, 'total_time': 0.098559925}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f8d-207f-74d0-90d4-caeba37058f5-0', tool_calls=[{'name': 'send_e

### Edit

In [93]:
agent = create_agent(
    model='groq:llama-3.3-70b-versatile',
    checkpointer=InMemorySaver(),
    tools=[read_email_tool,send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                'send_email_tool':{
                    'allowed_decisions':['approve','edit','reject']
                },
                'read_email_tool':False
            },
        )
    ]
)
config = {"configurable":{'thread_id':"test-edit"}}

In [94]:
result = agent.invoke({'messages':[HumanMessage(content="Send email to rahul@gmail.com with subject as Hello and body as how are you")]},config)

In [95]:
result

{'messages': [HumanMessage(content='Send email to rahul@gmail.com with subject as Hello and body as how are you', additional_kwargs={}, response_metadata={}, id='ab989f50-7507-4da2-9e88-5e53a56806f3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pfk4aqsdg', 'function': {'arguments': '{"body":"how are you","recipient":"rahul@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 301, 'total_tokens': 334, 'completion_time': 0.062792864, 'completion_tokens_details': None, 'prompt_time': 0.014766749, 'prompt_tokens_details': None, 'queue_time': 0.159554884, 'total_time': 0.077559613}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f91-e9de-78f2-9a3d-0026e9a3b084-0', tool_calls=[{'name': 'send_email_tool', 'args':

In [96]:
if "__interrupt__" in result:
    print("Paused Editing")

    res = agent.invoke(
        Command(
            resume={
                'decisions':[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"rahul@outlook.com",
                                "subject":"Corrected Subject",
                                "body":"This was edited by human"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    print(res)
    print(f"Result: {res['messages'][-1].content}")

Paused Editing
{'messages': [HumanMessage(content='Send email to rahul@gmail.com with subject as Hello and body as how are you', additional_kwargs={}, response_metadata={}, id='ab989f50-7507-4da2-9e88-5e53a56806f3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pfk4aqsdg', 'function': {'arguments': '{"body":"how are you","recipient":"rahul@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 301, 'total_tokens': 334, 'completion_time': 0.062792864, 'completion_tokens_details': None, 'prompt_time': 0.014766749, 'prompt_tokens_details': None, 'queue_time': 0.159554884, 'total_time': 0.077559613}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4f91-e9de-78f2-9a3d-0026e9a3b084-0', tool_calls=[{'type': 'tool_call', 